# JetRacer - Tune bam vach + Lai tay + Thu data train

Mot notebook lam ba viec:

1. **Xem camera & chinh nguong** bam vach dut o giua.
2. **Lai bang tay cam** de thu data train model (CV van chay song song de so sanh).
3. **Cho CV tu chay** de xem no da on chua.

**Truoc moi lan Run All**: `Kernel > Restart & Clear Output`. Hai notebook mo camera cung luc se bao `Failed to create CaptureSession`. Restart cung la cach duy nhat de code moi copy len xe co hieu luc (Python cache module da import).

**AN TOAN**

- Mo len la trang thai DUNG. Khong lenh nao xuong phan cung cho den khi bam **LAI TAY** hoac **CHAY**.
- **LAI TAY** va **CHAY** loai tru nhau - khong bao gio chay cung luc.
- Vao LAI TAY bi tu choi neu can gat chua ve giua.
- Vao CHAY bi tu choi neu dang mat vach tren 20% frame.
- **DUNG KHAN CAP** cat ga ngay. Camera dung hoac loi -> tu dong cat ga.
- Lan dau: **ke banh khoi mat dat**.

## 1. Kiem tra thu muc va file

In [ ]:
%cd /home/jetson/JetsonRacer

import os
import socket

required = [
    'tools/tune_lane_jupyter.py',
    'src/jetracer_baseline/tuning_ui.py',
    'src/jetracer_baseline/perception/lane.py',
    'src/jetracer_baseline/perception/shading.py',
    'configs/default.yaml',
]
missing = [p for p in required if not os.path.exists(p)]
print('Hostname:', socket.gethostname())
print('Thu muc:', os.getcwd())
print('Files:', 'OK' if not missing else 'THIEU ' + ', '.join(missing))
if missing:
    raise IOError('Chua copy du code moi sang Jetson')

print('Shading da hieu chuan:', os.path.exists('configs/shading.yaml'))

## 2. Mo giao dien

Thu tu:

1. **1. MO CAMERA**
2. Tab **Tay cam + Data** -> bam **KET NOI TAY CAM**. Neu bao CHUA KET NOI: **bam/xoay can mot cai** roi bam lai (trinh duyet chi gui su kien gamepad sau thao tac dau tien - day la hanh vi cua Gamepad API, khong phai loi).
3. Kiem tra `Truc lai` / `Truc ga` doc ra dung khi ban gat can. Sai thi doi so truc hoac tick `Dao lai` / `Dao ga`.
4. **KE BANH**, bam **LAI TAY**, thu gat can xem banh quay dung chieu.
5. Chinh mask o tab **Bam vach** cho sach (xem muc 4).
6. Dat xe xuong, bam **GHI DATA (train)** roi lai vai vong.
7. Muon xem CV tu chay: bam **DUNG LAI TAY** truoc, roi bam **CHAY**.

In [ ]:
%run tools/tune_lane_jupyter.py

## 3. Thu data de train model

Bam **GHI DATA (train)** -> ghi vao `data/driving/<session>_<gio>/`:

```
images/frame_000000.jpg ...
labels.csv
metadata.json
```

`labels.csv` co 20 cot. Quan trong nhat:

| Cot | Y nghia |
|---|---|
| `steering_cmd`, `throttle_cmd` | **Nhan de train** - lenh NGUOI lai |
| `cv_steer`, `cv_throttle` | Lenh CV truyen thong **tren cung frame** |
| `cte`, `curvature`, `drive_mode`, `n_bands`, `lane_found` | Trang thai CV luc do |

Nho cap `steering_cmd` vs `cv_steer` tren cung frame, so sanh duoc ngay **CV lai khac nguoi bao nhieu** ma khong phai chay lai lan hai. Cho nao lech lon la cho CV dang sai - do chinh la cac frame dang gia nhat de train.

Anh luu la frame **THO** (truoc sua mau va resize) nen train lai duoc voi bat ky tien xu ly nao.

**Thu theo session rieng** (doi `Session:` truoc moi lan ghi): `giua`, `lech_trai`, `lech_phai`, `toi`, `cua_gat`. Chia train/val/test phai chia THEO SESSION, khong random frame.

Muon xem duong di de toi uu: bam them **GHI VIDEO** -> `logs/tune_<gio>.avi` + `.sidecar.csv`. Gui ca hai ve cho toi.

## 4. Chan doan nhanh

| Nhin thay | Keo slider nao |
|---|---|
| Mask day dom trang | Tang `Bao hoa toi thieu (S)` hoac `Dien tich blob toi thieu` |
| Mask trong, banner bao MAT VACH | Giam `Bao hoa toi thieu (S)` va `Dien tich blob toi thieu` |
| Duong do bam vao vien lane | Giam `Be rong cum toi da` |
| `dai` chi 1-2 | Giam `Dien tich blob toi thieu` va `Pixel toi thieu / dai` |
| Nen van phong lot vao mask | Tang `ROI tren` |
| **Goc cua qua RONG, be khong du gat** | Xem muc 6 ben duoi - phai noi hard-limit cua driver |
| Cat cua, khong bam kip vach | Tang `Lai theo do cong`, tang `Boi lai khi cua` |
| Vao cua con nhanh qua | Giam `Ga khi vao CUA`, ha `Nguong VAO cua` |
| Doan thang di cham qua | Tang `Ga doan THANG` |
| Banner nhay THANG/CUA lien tuc | Ha `Nguong RA cua` cho xa `Nguong VAO cua` hon |
| Lai dao dong | Giam `PID Kp`, giam `Trong so diem ngam` |

## 5. Xem thu ma chac chan banh khong quay

Chi can khi muon dat xe tren ban ma van yen tam bam nut. Binh thuong KHONG can chay cell nay.

In [ ]:
try:
    ui.close()
except NameError:
    pass

from tools.tune_lane_jupyter import launch
ui = launch(driver_kind='dryrun')

## 5. Chay luot day du bang config vua luu

Giao dien tune KHONG ghi log CSV. So lieu chinh thuc phai lay tu mot luot chay day du qua CLI - do moi la con so dua vao Technical Paper.

In [ ]:
!python3 -m src.jetracer_baseline.cli run --task speed --driver nvidia \n    --override configs/tuned.yaml --max-seconds 60 --record

## 6. Dong truoc khi tat notebook

In [ ]:
ui.close()

## 6. Goc cua qua rong - cach be gat toi da

Banner hien `lai-0.60 servo+0.39 TRAN LAI` nghia la: bo dieu khien doi lai 0.60 nhung servo chi quay duoc 0.39 tren thang do 1.0 - tuc **chi dung 39% tam quay** cua servo. Banh khong the be gat hon cho den khi noi gioi han.

Ba gioi han NHAN DON nhau:

```
steer_max 0.60  x  driver.steering_gain 0.65  =  0.390
roi con bi chan boi driver.steering_output_max 0.40
```

**Buoc 1 - DO gioi han co khi that (BAT BUOC, xe KE BANH):**

```
python3 tools/check_hardware.py --driver nvidia --calibrate-steering --wheels-are-lifted
```

Tool quet cham output servo ra tung ben. **Nhin that ky banh truoc**, bam `Ctrl+C` NGAY KHI banh vua cham gam/hoc banh. Dong `output=` in ngay TRUOC dong cuoi la gia tri con an toan.

Khong duoc doan so nay: servo dam vao chan co khi se stall, chay servo hoac sut nguon ca Jetson.

**Buoc 2 - dat vao `configs/default.yaml`:**

```yaml
control:
  steer_max: 1.0
  corner_steer_max: 1.0
  driver:
    steering_gain: -1.0
    steering_output_min: -0.85   # thay bang so DO DUOC o buoc 1
    steering_output_max:  0.85
```

**Buoc 3 - kiem tra:** mo lai giao dien, vao cua thi banner phai hien `servo+0.85` va **khong con** chu `TRAN LAI`.

Neu chi muon cua gat ma doan thang van em: giu `steer_max` thap (0.6) va chi nang `corner_steer_max` len 1.0.